In [1]:
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .enableHiveSupport()
    .appName("cu2")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
sql = spark.sql


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 15:20:04 WARN Utils: Service 'sparkDriver' could not bind on port 23527. Attempting port 23528.
26/05/19 15:20:04 WARN Utils: Service 'sparkDriver' could not bind on port 23528. Attempting port 23529.
26/05/19 15:20:04 WARN Utils: Service 'sparkDriver' could not bind on port 23529. Attempting port 23530.
26/05/19 15:20:04 WARN Utils: Service 'sparkDriver' could not bind on port 23530. Attempting port 23531.
26/05/19 15:20:15 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 23527. Attempting port 23528.
26/05/19 15:20:15 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 23528. Attempting port 23529.
26/05/19 15:20:15 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 23529. Attempting port 23530.


In [2]:
df = pd.read_csv(
    "../data/Referentiel_CIM-10-20250108.csv", sep=";", header=1, dtype=str
)


df = df[["code", "libellé long", "code MCO/HAD"]].rename(
    columns={"libellé long": "definition", "code MCO/HAD": "mco_had"}
)

df["code"] = df["code"].astype(str).str.strip()
# df['definition'] = df['definition'].astype(str).str.lower()
df["mco_had"] = pd.to_numeric(df["mco_had"], errors="coerce")
df = df[df["mco_had"].isin([0])].drop("mco_had", axis=1)

dp_codes=df["code"].to_list()
print(len(dp_codes))
#dp_codes

13358


In [3]:
syn = pd.read_pickle("../../binary_model/data/V2_avril_2026/phase1_df_pos.pkl")
syn

,code,definition,libelle,label
0,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera a vibrio cholerae 01, biovar cholerae",1
1,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera, classique",1
2,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera asiatique, classique",1
3,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera epidemique, classique",1
4,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera malin, classique",1
...,...,...,...,...
156872,Z998,dependance envers d'autres machines et apparei...,"dependance vis-a-vis, appareil, precise nca",1
156873,Z998,dependance envers d'autres machines et apparei...,"dependance vis-a-vis, appareil, auxiliaire, pr...",1
156874,Z999,dependance envers une machine et un appareil a...,"dependance vis-a-vis, appareil",1
156875,Z999,dependance envers une machine et un appareil a...,dependance envers une machine et un appareil a...,1


In [4]:
syn = syn[syn['code'].isin(dp_codes)]
syn

,code,definition,libelle,label
0,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera a vibrio cholerae 01, biovar cholerae",1
1,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera, classique",1
2,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera asiatique, classique",1
3,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera epidemique, classique",1
4,A000,"cholera a vibrio cholerae 01, biovar cholerae","cholera malin, classique",1
...,...,...,...,...
156872,Z998,dependance envers d'autres machines et apparei...,"dependance vis-a-vis, appareil, precise nca",1
156873,Z998,dependance envers d'autres machines et apparei...,"dependance vis-a-vis, appareil, auxiliaire, pr...",1
156874,Z999,dependance envers une machine et un appareil a...,"dependance vis-a-vis, appareil",1
156875,Z999,dependance envers une machine et un appareil a...,dependance envers une machine et un appareil a...,1


In [5]:
df = spark.createDataFrame(syn) \
    .withColumnRenamed("libelle", "note_text") \
    .withColumnRenamed("code", "dp")

In [6]:
labels_dp = df.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
print(len(labels_dp))

[Stage 1:==================================================> (1961 + 20) / 2000]

13328


In [7]:
from pyspark.sql import functions as F

def save_df_to_parquet(df, filename, num_partitions):
    df = df.withColumn("note_id", F.monotonically_increasing_id())

    df_to_save = df.select(
        "note_id",
        F.col("note_text"),
        F.col("dp")
    )

    df_to_save.repartition(num_partitions).write.mode("overwrite").parquet(filename)


save_df_to_parquet(df, "CU2/encoder_baseline/syn", num_partitions=121)


In [8]:
import subprocess
user="ldedieu"

subprocess.run(
    #["mkdir", "-p", f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2"],
    ["mkdir", "-p", f"/data/scratch/{user}/CU2/encoder_baseline/syn"],
    check=True
)

subprocess.run(
    [
        "hdfs", "dfs", "-copyToLocal", "-f",
        #f"ia_codage/dataset_800k/{strat}_{n_class_dp}class_v2/*",
        #f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2/"
        f"CU2/encoder_baseline/syn/*",
        f"/data/scratch/{user}/CU2/encoder_baseline/syn/"
    ],
    check=True
)

CompletedProcess(args=['hdfs', 'dfs', '-copyToLocal', '-f', 'CU2/encoder_baseline/syn/*', '/data/scratch/ldedieu/CU2/encoder_baseline/syn/'], returncode=0)